# Tô Màu Bản Đồ Việt Nam — Thuật Toán AC-3

**Bài toán:** Tô màu bản đồ sao cho **không có hai tỉnh nào kề nhau có cùng màu**.

**Thuật toán:** **AC-3** (Arc Consistency Algorithm #3) — lan truyền ràng buộc (constraint propagation)
để thu hẹp miền giá trị trước khi gán màu, giúp giảm không gian tìm kiếm.

**Ý tưởng chính:**
1. Với mỗi cặp tỉnh kề nhau (Xi, Xj), kiểm tra: mỗi màu trong miền của Xi có ít nhất một màu khác trong miền của Xj không?
2. Nếu không, loại bỏ màu đó khỏi miền của Xi.
3. Khi miền của Xi bị thu hẹp, kiểm tra lại tất cả các tỉnh kề với Xi.
4. Lặp lại cho đến khi không còn thay đổi nào.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque
import numpy as np

plt.rcParams['font.family'] = 'Tahoma'

## 1. Ví dụ đơn giản — Minh họa AC-3 từng bước

Xét 3 tỉnh A, B, C tạo thành **tam giác** (mỗi cặp đều kề nhau). Chỉ có **3 màu**: Đỏ, Xanh dương, Xanh lá.

In [ ]:
# ── Dữ liệu ví dụ đơn giản ──
SIMPLE_ADJ = {
    'A': ['B', 'C'],
    'B': ['A', 'C'],
    'C': ['A', 'B'],
}

SIMPLE_POS = {'A': (0, 1.2), 'B': (-1, 0), 'C': (1, 0)}

COLORS_3 = ['Đỏ', 'Xanh dương', 'Xanh lá']
HEX_3 = {'Đỏ': '#E74C3C', 'Xanh dương': '#3498DB', 'Xanh lá': '#27AE60'}

In [ ]:
def revise(domains, xi, xj):
    """Loại bỏ giá trị khỏi domain[xi] nếu không có giá trị hỗ trợ trong domain[xj]."""
    revised = False
    for x in sorted(domains[xi]):
        if all(x == y for y in domains[xj]):
            domains[xi].discard(x)
            revised = True
    return revised


def ac3(domains, adjacency, verbose=True):
    """Thuật toán AC-3 — Lan truyền ràng buộc đạt Arc Consistency."""
    # Khởi tạo hàng đợi với tất cả các cung (Xi → Xj)
    queue = deque()
    for xi in adjacency:
        for xj in adjacency[xi]:
            queue.append((xi, xj))

    step = 0
    reductions = 0

    if verbose:
        print(f"Hàng đợi khởi tạo: {len(queue)} cung")
        print("Miền giá trị ban đầu:")
        for p, d in sorted(domains.items()):
            print(f"  {p}: {{{', '.join(sorted(d))}}}")
        print()

    while queue:
        xi, xj = queue.popleft()
        step += 1
        old = set(domains[xi])

        if revise(domains, xi, xj):
            removed = old - domains[xi]
            reductions += 1
            if verbose:
                print(f"Bước {step}: Cung ({xi} → {xj})")
                print(f"  {xi}: {{{', '.join(sorted(old))}}} → {{{', '.join(sorted(domains[xi]))}}}")
                print(f"  Loại bỏ: {{{', '.join(sorted(removed))}}}")
                print(f"  Lý do: domain[{xj}]={{{', '.join(sorted(domains[xj]))}}} không có giá trị ≠ {{{', '.join(sorted(removed))}}}")

            if len(domains[xi]) == 0:
                if verbose:
                    print(f"  ❌ Miền {xi} rỗng → Mâu thuẫn!")
                return False

            for xk in adjacency[xi]:
                if xk != xj:
                    queue.append((xk, xi))

    if verbose:
        print(f"✅ AC-3 hoàn tất: {step} bước kiểm tra, {reductions} lần thu hẹp miền")
        print("Miền giá trị sau AC-3:")
        for p, d in sorted(domains.items()):
            print(f"  {p}: {{{', '.join(sorted(d))}}}")
        print()
    return True


def draw_constraint_graph(positions, adjacency, domains, assignment, title, hex_map):
    """Vẽ đồ thị ràng buộc với nút được tô màu."""
    fig, ax = plt.subplots(figsize=(5.5, 4.5))

    drawn = set()
    for u in adjacency:
        for v in adjacency[u]:
            if (v, u) not in drawn:
                drawn.add((u, v))
                x1, y1 = positions[u]
                x2, y2 = positions[v]
                ax.plot([x1, x2], [y1, y2], 'gray', linewidth=1.5, alpha=0.5, zorder=1)

    for node, (x, y) in positions.items():
        assigned = node in assignment
        color = hex_map[assignment[node]] if assigned else '#E0E0E0'
        circle = mpatches.Circle((x, y), 0.28, facecolor=color, edgecolor='black',
                           linewidth=2.5, zorder=5)
        ax.add_patch(circle)

        if assigned:
            label = f"{node}\n({assignment[node]})"
        else:
            d = domains[node] if node in domains else set()
            label = f"{node}\n{{{','.join(sorted(d))}}}" if d else f"{node}"
        ax.text(x, y - 0.48, label, ha='center', va='top', fontsize=8.5, fontweight='bold')

    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-0.8, 2.0)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)

    legend = [mpatches.Patch(color=c, label=n) for n, c in hex_map.items()]
    legend.append(mpatches.Patch(color='#E0E0E0', label='Chưa gán'))
    ax.legend(handles=legend, loc='lower right', fontsize=7.5)
    plt.tight_layout()
    plt.show()

In [ ]:
print("=" * 55)
print("VÍ DỤ ĐƠN GIẢN: 3 tỉnh (tam giác) — 3 màu")
print("=" * 55)

# Bước 0: Khởi tạo miền giá trị (chưa gán gì)
dom0 = {v: set(COLORS_3) for v in SIMPLE_ADJ}

draw_constraint_graph(SIMPLE_POS, SIMPLE_ADJ, dom0, {},
                     'Bước 0: Miền giá trị ban đầu (chưa gán)', HEX_3)

print("AC-3 trên miền ban đầu (chưa có ràng buộc cụ thể):")
ac3(dom0, SIMPLE_ADJ)
print("→ Chưa có thay đổi vì mỗi miền có 3 màu, luôn tìm được màu khác.\n")

# Bước 1: Gán A = Đỏ, chạy AC-3
print("─" * 55)
print("Bước 1: GÁN A = Đỏ → chạy AC-3 lan truyền ràng buộc")
print("─" * 55)
dom1 = {v: set(COLORS_3) for v in SIMPLE_ADJ}
dom1['A'] = {'Đỏ'}
ac3(dom1, SIMPLE_ADJ)

draw_constraint_graph(SIMPLE_POS, SIMPLE_ADJ, dom1, {'A': 'Đỏ'},
                     'Bước 1: Gán A=Đỏ + AC-3', HEX_3)

# Bước 2: Gán B = Xanh dương, chạy AC-3
print("─" * 55)
print("Bước 2: GÁN B = Xanh dương → chạy AC-3")
print("─" * 55)
dom2 = {v: set(dom1[v]) for v in dom1}
dom2['B'] = {'Xanh dương'}
ac3(dom2, SIMPLE_ADJ)

draw_constraint_graph(SIMPLE_POS, SIMPLE_ADJ, dom2,
                     {'A': 'Đỏ', 'B': 'Xanh dương'},
                     'Bước 2: Gán B=Xanh dương + AC-3', HEX_3)

print("=" * 55)
print(f"✅ KẾT QUẢ: A=Đỏ, B=Xanh dương, C={list(dom2['C'])[0]}")
print("=" * 55)

## 2. Tô màu bản đồ Việt Nam

Áp dụng AC-3 kết hợp **tìm kiếm quay lui** (backtracking) để tô màu **22 tỉnh/thành**
của Việt Nam với **4 màu**. Các tỉnh được chọn trải dài từ Bắc vào Nam,
bao gồm cả Tây Nguyên và Đồng bằng Sông Cửu Long.

**Màu sử dụng:** Đỏ, Xanh dương, Xanh lá, Vàng

In [ ]:
# ── Dữ liệu bản đồ Việt Nam ──
# Mỗi tỉnh: (tên đầy đủ, tọa độ x, tọa độ y)
# Tọa độ mô phỏng vị trí địa lý: x ≈ kinh độ, y ≈ vĩ độ

PROVINCES = {
    'HG':  ('Hà Giang',          4.5, 18.5),
    'CB':  ('Cao Bằng',          7.0, 18.0),
    'LC':  ('Lào Cai',           2.5, 18.0),
    'QNH': ('Quảng Ninh',        9.0, 17.0),
    'HN':  ('Hà Nội',            5.0, 16.5),
    'HP':  ('Hải Phòng',         8.5, 16.0),
    'TH':  ('Thanh Hóa',         5.5, 15.0),
    'NA':  ('Nghệ An',           4.5, 14.0),
    'HT':  ('Hà Tĩnh',           5.0, 13.0),
    'QB':  ('Quảng Bình',        5.5, 12.0),
    'TTH': ('Thừa Thiên Huế',    6.5, 10.5),
    'DN':  ('Đà Nẵng',           8.0, 10.0),
    'QNM': ('Quảng Nam',         7.0,  9.5),
    'QNG': ('Quảng Ngãi',        8.0,  8.5),
    'BD':  ('Bình Định',         8.5,  7.0),
    'KH':  ('Khánh Hòa',         8.5,  5.0),
    'KT':  ('Kon Tum',           5.5,  9.0),
    'GL':  ('Gia Lai',           6.0,  7.5),
    'DL':  ('Đắk Lắk',           6.5,  6.0),
    'HCM': ('TP. Hồ Chí Minh',   6.5,  2.5),
    'CT':  ('Cần Thơ',           5.5,  1.0),
    'CM':  ('Cà Mau',            5.0,  0.0),
}

# Quan hệ kề nhau (dựa trên địa lý thực tế)
ADJACENCY = {
    'HG':  ['CB', 'LC', 'HN'],
    'CB':  ['HG', 'QNH', 'HN'],
    'LC':  ['HG'],
    'QNH': ['CB', 'HN', 'HP'],
    'HN':  ['HG', 'CB', 'QNH', 'HP', 'TH'],
    'HP':  ['QNH', 'HN', 'TH'],
    'TH':  ['HN', 'HP', 'NA'],
    'NA':  ['TH', 'HT'],
    'HT':  ['NA', 'QB'],
    'QB':  ['HT', 'TTH', 'KT'],
    'TTH': ['QB', 'DN', 'QNM'],
    'DN':  ['TTH', 'QNM'],
    'QNM': ['TTH', 'DN', 'QNG', 'KT'],
    'QNG': ['QNM', 'BD', 'KT'],
    'BD':  ['QNG', 'GL', 'KH'],
    'KH':  ['BD', 'DL'],
    'KT':  ['QB', 'QNM', 'QNG', 'GL'],
    'GL':  ['BD', 'KT', 'DL'],
    'DL':  ['KH', 'GL', 'HCM'],
    'HCM': ['DL', 'CT'],
    'CT':  ['HCM', 'CM'],
    'CM':  ['CT'],
}

# 4 màu cho bản đồ
COLORS_4 = ['Đỏ', 'Xanh dương', 'Xanh lá', 'Vàng']
HEX_4 = {
    'Đỏ':          '#E74C3C',
    'Xanh dương':  '#3498DB',
    'Xanh lá':     '#27AE60',
    'Vàng':        '#F1C40F',
}

In [ ]:
def draw_vietnam_map(provinces, adjacency, assignment, title, hex_map):
    """Vẽ bản đồ Việt Nam dạng đồ thị ràng buộc."""
    fig, ax = plt.subplots(figsize=(8, 14))

    # Vẽ đường viền dáng Việt Nam (nền nhạt)
    outline_x = [9.5, 9.3, 9.0, 6.0, 5.5, 5.0, 4.0, 2.0, 2.5, 3.5, 5.0,
                 6.0, 6.5, 7.0, 8.5, 9.5, 9.8, 9.5, 9.0, 8.0, 7.5, 7.0,
                 6.0, 5.0, 4.5, 5.5, 5.0, 4.0, 2.5]
    outline_y = [16.5, 17.5, 18.5, 19.0, 18.5, 18.0, 18.5, 17.5, 16.0,
                 15.5, 14.5, 13.5, 11.5, 10.5, 9.5, 8.5, 7.5, 6.5, 5.5,
                 4.5, 3.5, 3.0, 2.5, 1.5, 0.5, -0.5, -0.5, 0.5, 1.5]
    ax.fill(outline_x, outline_y, facecolor='#F5F0E8', edgecolor='#AAAAAA',
            linewidth=1.5, alpha=0.5, zorder=0)

    # Vẽ cạnh (ràng buộc kề nhau)
    drawn = set()
    for u in adjacency:
        for v in adjacency[u]:
            if (v, u) not in drawn:
                drawn.add((u, v))
                x1, y1 = provinces[u][1], provinces[u][2]
                x2, y2 = provinces[v][1], provinces[v][2]
                ax.plot([x1, x2], [y1, y2], 'gray', linewidth=1.2,
                       alpha=0.45, zorder=2)

    # Vẽ nút (tỉnh)
    for code, (name, x, y) in provinces.items():
        assigned = code in assignment
        color = hex_map[assignment[code]] if assigned else '#E0E0E0'
        circle = mpatches.Circle((x, y), 0.35, facecolor=color, edgecolor='black',
                           linewidth=2, zorder=5)
        ax.add_patch(circle)

        # Nhãn: tên viết tắt + tên đầy đủ
        label = f"{code}" if assigned else f"{code}"
        ax.text(x, y, label, ha='center', va='center', fontsize=6.5,
               fontweight='bold', color='white' if assigned else '#333333', zorder=6)
        ax.text(x, y - 0.52, name, ha='center', va='top', fontsize=6,
               fontweight='normal', color='#333333')

    ax.set_xlim(1.2, 10.5)
    ax.set_ylim(-1.2, 19.8)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)

    legend = [mpatches.Patch(color=c, label=n) for n, c in hex_map.items()]
    legend.append(mpatches.Patch(color='#E0E0E0', label='Chưa tô'))
    ax.legend(handles=legend, loc='lower right', fontsize=7.5, ncol=3)
    plt.tight_layout()
    plt.show()

In [ ]:
def backtrack_ac3(provinces, domains, adjacency, verbose=True):
    """Tìm kiếm quay lui kết hợp AC-3 để tô màu bản đồ."""

    # 1. Chạy AC-3 để lan truyền ràng buộc
    if not ac3(domains, adjacency, verbose=False):
        return None

    # 2. Kiểm tra xem đã gán hết chưa
    unassigned = [p for p in provinces if len(domains[p]) > 1]
    if not unassigned:
        return {p: list(domains[p])[0] for p in provinces}

    # 3. Chọn biến có miền nhỏ nhất (MRV heuristic)
    var = min(unassigned, key=lambda p: len(domains[p]))

    if verbose:
        print(f"  Chọn {var} ({provinces[var][0]}): miền={{{', '.join(sorted(domains[var]))}}}")

    # 4. Thử từng màu trong miền của var
    for value in sorted(domains[var]):
        if verbose:
            print(f"    Thử gán {var} = {value}...", end=' ')

        new_domains = {p: set(domains[p]) for p in provinces}
        new_domains[var] = {value}

        result = backtrack_ac3(provinces, new_domains, adjacency, verbose=False)
        if result:
            if verbose:
                print("✅")
            return result
        else:
            if verbose:
                print("❌ thất bại, thử màu khác")

    if verbose:
        print(f"    ← Quay lui khỏi {var} (không màu nào hợp lệ)")
    return None

In [ ]:
# ── Chạy thuật toán tô màu bản đồ Việt Nam ──

print("=" * 60)
print("TÔ MÀU BẢN ĐỒ VIỆT NAM — AC-3 + BACKTRACKING")
print(f"Số tỉnh: {len(PROVINCES)} | Số màu: {len(COLORS_4)}")
print("=" * 60)

# Vẽ bản đồ trước khi tô màu
draw_vietnam_map(PROVINCES, ADJACENCY, {},
                'Bản đồ Việt Nam — Trước khi tô màu', HEX_4)

# Khởi tạo miền giá trị
initial_domains = {p: set(COLORS_4) for p in PROVINCES}

print("\nQuá trình tìm kiếm quay lui + AC-3:\n")

# Chạy solver
solution = backtrack_ac3(PROVINCES, initial_domains, ADJACENCY)

print("\n" + "=" * 60)
if solution:
    print("✅ TÔ MÀU THÀNH CÔNG!")
    print("=" * 60)
    print("\nKết quả tô màu từng tỉnh:")
    # Nhóm theo màu
    by_color = {}
    for code, color in sorted(solution.items()):
        by_color.setdefault(color, []).append(PROVINCES[code][0])
    for color in COLORS_4:
        if color in by_color:
            print(f"  {color}: {', '.join(by_color[color])}")

    # Kiểm tra ràng buộc
    violations = 0
    for u in ADJACENCY:
        for v in ADJACENCY[u]:
            if solution[u] == solution[v]:
                violations += 1
                print(f"  ⚠ Vi phạm: {PROVINCES[u][0]} và {PROVINCES[v][0]} cùng màu {solution[u]}")
    print(f"\nSố vi phạm ràng buộc: {violations}")

    # Vẽ bản đồ đã tô màu
    draw_vietnam_map(PROVINCES, ADJACENCY, solution,
                    'Bản đồ Việt Nam — Sau khi tô màu (AC-3)', HEX_4)
else:
    print("❌ KHÔNG TÌM THẤY LỜI GIẢI")
    print("=" * 60)

## 3. Tổng kết

| Thành phần | Mô tả |
|------------|-------|
| **Biến** | 22 tỉnh/thành của Việt Nam |
| **Miền giá trị** | {Đỏ, Xanh dương, Xanh lá, Vàng} |
| **Ràng buộc** | Hai tỉnh kề nhau phải khác màu |
| **AC-3** | Lan truyền ràng buộc, thu hẹp miền trước khi gán |
| **Tìm kiếm** | Quay lui (backtracking) + MRV |

**Ưu điểm của AC-3:**
- Phát hiện sớm các giá trị không khả thi, giảm không gian tìm kiếm
- Lan truyền hiệu quả: khi một biến bị thu hẹp, chỉ kiểm tra lại các biến liên quan
- Độ phức tạp: O(n × d³) với n = số biến, d = kích thước miền

**Định lý 4 màu:** Mọi bản đồ phẳng đều có thể tô bằng **tối đa 4 màu**.